# Preprocessing steps

## Loading and normalizing the data

In [1]:
dataset_path = "../dataset/bigcodebench_normalized.json"
filtered_dataset_path = "../dataset/bigcodebench_normalized_filtered.json"

In [2]:
from datasets import load_dataset 
ds = load_dataset("bigcode/bigcodebench") 
split_name = list(ds.keys())[-1]
print("Using split:", split_name)
dataset_split = ds[split_name] 
print(dataset_split[0].keys())

Using split: v0.1.4
dict_keys(['task_id', 'complete_prompt', 'instruct_prompt', 'canonical_solution', 'code_prompt', 'test', 'entry_point', 'doc_struct', 'libs'])


In [3]:
from src.utils import normalize
import json

normalized_data = normalize(dataset_split)

print(json.dumps(normalized_data[0], indent=2))

with open(dataset_path, "w", encoding="utf-8") as f:
        json.dump(normalized_data, f, indent=2, ensure_ascii=False)

print(f"Normalized dataset saved as {dataset_path}")

{
  "id": "BigCodeBench/0",
  "language": "python",
  "original_code": "import itertools\nfrom random import shuffle\n\ndef task_func(numbers=list(range(1, 3))):\n    permutations = list(itertools.permutations(numbers))\n    sum_diffs = 0\n\n    for perm in permutations:\n        perm = list(perm)\n        shuffle(perm)\n        diffs = [abs(perm[i] - perm[i+1]) for i in range(len(perm)-1)]\n        sum_diffs += sum(diffs)\n\n    avg_sum_diffs = sum_diffs / len(permutations)\n    \n    return avg_sum_diffs",
  "test": [
    "import unittest\nfrom unittest.mock import patch\nfrom random import seed, shuffle\nimport itertools\nclass TestCases(unittest.TestCase):\n    def test_default_numbers(self):\n        # Test with default number range (1 to 10) to check that the result is a positive float.\n        result = task_func()\n        self.assertIsInstance(result, float)\n        self.assertGreater(result, 0)\n    def test_custom_list(self):\n        # Test with a custom list of small posi

In [4]:
# marking hard and easy problems
from datasets import load_dataset 
from src.utils import mark_hard_easy
import json

ds_hard =load_dataset("bigcode/bigcodebench-hard")
split_name_hard = list(ds_hard.keys())[-1]
with open(dataset_path, "r", encoding="utf-8") as f:
        normalized_data = json.load(f) 
dataset_hard_split = ds_hard[split_name_hard] 
marked_all = mark_hard_easy(normalized_data, dataset_hard_split,dataset_path) 

## Running tests

In [5]:
# # installing missing packages
# from src.utils import install_package, extract_required_packages
# with open(dataset_path, "r", encoding="utf-8") as f:
#         data = json.load(f)
# packages = extract_required_packages(data)
# print(f"Requried packages: {packages}")
# for pkg in packages:
#     try:
#         __import__(pkg)
#     except ImportError:
#         try:
#             print(f"Installing missing package: {pkg}")
#             install_package(pkg)
#         except Exception as e:
#             print(f"⚠️ Could not install package {pkg}, skipping. Reason: {e}")

In [6]:
# # to run the original tests
# from src.utils import run_original_tests
# import json
# dataset_path = "../dataset/bigcodebench_normalized.json"
# output_file="../results/original_test_results.json" 
# with open(dataset_path, "r", encoding="utf-8") as f:
#         data = json.load(f)
# run_original_tests(data, output_file)
# # not running [205, 363] because of multiprocessing

In [7]:
from src.utils import analyze_test_results
import json

with open("../results/original_test_results.json", "r", encoding="utf-8") as f:
    data = json.load(f)

summary = analyze_test_results(data)

print(json.dumps(summary, indent=2))

{
  "total_pass": 6080,
  "total_fail": 245,
  "num_entries_with_failures": 75,
  "num_entries_all_fail": 44,
  "entries_with_failures": [
    "BigCodeBench/14",
    "BigCodeBench/15",
    "BigCodeBench/18",
    "BigCodeBench/39",
    "BigCodeBench/49",
    "BigCodeBench/63",
    "BigCodeBench/73",
    "BigCodeBench/101",
    "BigCodeBench/111",
    "BigCodeBench/115",
    "BigCodeBench/157",
    "BigCodeBench/177",
    "BigCodeBench/205",
    "BigCodeBench/219",
    "BigCodeBench/221",
    "BigCodeBench/227",
    "BigCodeBench/235",
    "BigCodeBench/237",
    "BigCodeBench/245",
    "BigCodeBench/265",
    "BigCodeBench/276",
    "BigCodeBench/288",
    "BigCodeBench/296",
    "BigCodeBench/313",
    "BigCodeBench/323",
    "BigCodeBench/334",
    "BigCodeBench/350",
    "BigCodeBench/352",
    "BigCodeBench/357",
    "BigCodeBench/363",
    "BigCodeBench/377",
    "BigCodeBench/383",
    "BigCodeBench/394",
    "BigCodeBench/407",
    "BigCodeBench/408",
    "BigCodeBench/416",
    

## Filtering based on tests and easy split

In [8]:
from src.utils import filter_dataset

filter_dataset(
    original_dataset_file= dataset_path,
    test_results_file="../results/original_test_results.json",
    output_file=filtered_dataset_path
)

Filtered dataset saved to ../dataset/bigcodebench_normalized_filtered.json
Original entries: 1140
Filtered entries (all tests pass + easy split): 0


## Random split for experiments 

In [9]:
from src.utils import sample_random_entries

sample_random_entries(
    input_file=filtered_dataset_path,
    experiment_output_file="../dataset/dataset.json",
    extension_output_file="../dataset/dataset_extension.json",
    sample_size=50,
    seed=0
)


Sampled 50 twice from 0 total.
Sampled dataset 1 saved to ../dataset/dataset.json
Sampled dataset 2 saved to ../dataset/dataset_extension.json
